# Section 2 Assignment - Automatic Text Generation

# Task
The task is to generate a number of versions of a short article entitled "The Implications of Recent Developments in Artificial Intelligence".

The exercise here it use:

- a variety of the decoding methods
- models and
- parameters

introduced through the course material to do this, while comparing and assessing the outputs.


Ideally, an article looks like:

- it was written by a human and looks credible,
- with reference to concrete facts,
- without using excessive computing.

It needs to be
- rich and informative, yet concise and complete.

Excessive diversity will result in jibberish, while restricted diversity will lead to repetition and will suggest a non-human auther.

You will need to consider how best to set up the prompts and the effects of increasing processing loads.

Consider, for each text generation choice,
- what the "giveaways" are that may suggest that the text is not written by a human.

When you see a good performance, show the effect of degrading it.

(Subsequently in the module I will introduce you to tools and metrics that will help with this, but for now I want to motivate your judgement of how such an NLU task would be assessed)


# Workbook format
- models used - gpt2-medium, qwen for comparison
- prompts
- decoding methods
- qwen comparison
- degradation


### Setup / Shared code
Uses example workbook code to create functions that are used to:

1. Build a model
1. Generate text using this model

In [1]:
import textwrap
from pprint import pprint
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 1000)

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def buildModel(model_name) :

  device = "cuda" if torch.cuda.is_available() else "cpu"


  # download + load tokenizer for the model
  tokenizer = AutoTokenizer.from_pretrained(model_name)

  # load the model , to predict text
  model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

  return  tokenizer,model, device

In [3]:
def generate_text(tokenizer,model,device, prompt,  **text_gen_kwargs):

    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)

    # do_sample = False means a greedy search
    output = model.generate(input_ids, **text_gen_kwargs)

    generated_text = tokenizer.decode(output[0])

    return { "prompt": prompt,"generated_text": generated_text, **text_gen_kwargs}

In [4]:
# https://huggingface.co/openai-community/gpt2-medium
# "gpt2-xl"
model_name = "gpt2-medium"

tokenizer,model, device = buildModel(model_name)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Prompt

In [9]:
input_text = "The following is a short article written for a technology magazine.  The Implications of Recent Developments in Artificial Intelligence"

## Decoding methods
Hugging face parameters - https://huggingface.co/docs/transformers/main_classes/text_generation

max_new_tokens = 200 for all decodings

### Greedy Generation
**Configuration 1:**

    "max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1

**Configuration 2**:

    "max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1.3

In the 2nd configuration, repetition_penality is set to 1.3 to discourage the repetition seen in the 1st message.

In [11]:

args = [
    {"max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1},
    {"max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1.3}
]

results = []
for config in args:
    result = generate_text(tokenizer,model, device,input_text, **config)
    results.append(result)

result_df = pd.DataFrame(results)
display (result_df)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,max_new_tokens,do_sample,repetition_penalty
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of Work and the Economy. \n\nThe Future of Work and the Economy\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them. The future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n",200,False,1.0
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Robotics\n\nIn the past few years, artificial intelligence has become increasingly sophisticated at understanding human behavior through machine learning algorithms that are able to learn from experience or observe patterns within data sets (e-learning). This means we can now use these techniques on our own devices without having any prior knowledge about how they work – which makes them ideal candidates as tools used to improve humans' abilities with their everyday lives. However this does not mean there will be no need if AI becomes more intelligent than us: it could even lead directly into an era where machines take over jobs such like driving cars by using advanced computer vision systems instead! In fact, one recent study suggests robots may soon replace many manual workers who currently do most tasks around factories today; however some e...",200,False,1.3


### Analysis of generated text

**Text generated using Configuration 1**
> The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of Work and the Economy. \n\nThe Future of Work and the Economy\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them. The future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n

**Text generated using Configuration 2**
> The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Robotics\n\nIn the past few years, artificial intelligence has become increasingly sophisticated at understanding human behavior through machine learning algorithms that are able to learn from experience or observe patterns within data sets (e-learning). This means we can now use these techniques on our own devices without having any prior knowledge about how they work – which makes them ideal candidates as tools used to improve humans' abilities with their everyday lives. However this does not mean there will be no need if AI becomes more intelligent than us: it could even lead directly into an era where machines take over jobs such like driving cars by using advanced computer vision systems instead! In fact, one recent study suggests robots may soon replace many manual workers who currently do most tasks around factories today; however some e...

**Observations**
1. The first text contained repeated text "The future of work is not a question whether we will have jobs...them."  The model, has identified this phrase/sentence as the most likely successor to the preceeding phrasing.
1. Both texts are generated with the same configuration exception the second text specifies a repetition_penality of 1.3 to discourage repetition. This produces a more coherent body of text.

**Giveaways**

1. The 1st text is full of repeating text
1. The 2nd text has phrasing:
    - "improve humans' abilities"
    - "computer vision systems instead!"

which does not look correct.

2. The 2nd text uses a colon rather than a semi colon in the phrase "than us: it", which I think is incorrect.

**Conclusion**
Configuration two produces a readable sentence, suggesting that the repetition_penalty can improve greedy decoding / the quality of the genereated text. However, some punctuation and phrasing in text 2 would suggest that it is computer generated.

### Beam Search

In [ ]:
args = {"do_sample":False, "max_length":  300,"num_beams":5}

# max_new_tokens=n_steps,
result = generate_text(tokenizer,model, device,input_text, **args)

result_df = pd.DataFrame([result])
display (result_df)

generate_text(tokenizer,model, device,input_text,**args )

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,num_beams
0,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence and Machine Learning for the Future of the U.S. Economy.""\n\nThe paper, co-authored by University of California, Berkeley, economics professor Erik Brynjolfsson and Harvard University professor Andrew McAfee, was published by the National Bureau of Economic Research.\n\nBrynjolfsson and McAfee argue that advances in artificial intelligence and machine learning will have a profound impact on the U.S. economy in the coming decades.\n\n""The implications of recent developments in artificial intelligence and machine learning for the future of the U.S. economy are profound,"" Brynjolfsson and McAfee write in the paper.\n\nThe authors argue that advances in artificial intelligence and machine learning will have a profound impact on the U.S. economy in the coming decades.\n\n""The implications of recent developments in artificial intelligence and machine learning for the future of the U.S. economy are profound,"" Brynjolfsson ...",False,300,5


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'prompt': 'The Implications of Recent Developments in Artificial Intelligence',
 'generated_text': 'The Implications of Recent Developments in Artificial Intelligence and Machine Learning for the Future of the U.S. Economy."\n\nThe paper, co-authored by University of California, Berkeley, economics professor Erik Brynjolfsson and Harvard University professor Andrew McAfee, was published by the National Bureau of Economic Research.\n\nBrynjolfsson and McAfee argue that advances in artificial intelligence and machine learning will have a profound impact on the U.S. economy in the coming decades.\n\n"The implications of recent developments in artificial intelligence and machine learning for the future of the U.S. economy are profound," Brynjolfsson and McAfee write in the paper.\n\nThe authors argue that advances in artificial intelligence and machine learning will have a profound impact on the U.S. economy in the coming decades.\n\n"The implications of recent developments in artificial 

In [ ]:


args = {"do_sample":False, "max_length":  300,"num_beams":5, "no_repeat_ngram_size":2}

# max_new_tokens=n_steps,
result = generate_text(tokenizer,model, device,input_text, **args)

result_df = pd.DataFrame([result])
display (result_df)

generate_text(tokenizer,model, device,input_text,**args )

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,num_beams,no_repeat_ngram_size
0,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence and Machine Learning for the Future of the U.S. Economy.""\n\nThe paper, co-authored by University of California, Berkeley, economics professor Erik Brynjolfsson and MIT professor Andrew McAfee, argues that advances in artificial intelligence and machine learning have the potential to dramatically change the way we do business, the economy, and society in the coming decades. The authors argue that these advances will have a profound impact on the future of work, education, health care, transportation, energy, finance, government, law, media, entertainment, politics, science, technology and the environment. They also predict that the impact of these changes will be felt most acutely by those who are least able to adapt to them, such as the elderly, people with disabilities, low-skilled workers, women, racial and ethnic minorities, immigrants and people living in rural areas.<|endoftext|>",False,300,5,2


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'prompt': 'The Implications of Recent Developments in Artificial Intelligence',
 'generated_text': 'The Implications of Recent Developments in Artificial Intelligence and Machine Learning for the Future of the U.S. Economy."\n\nThe paper, co-authored by University of California, Berkeley, economics professor Erik Brynjolfsson and MIT professor Andrew McAfee, argues that advances in artificial intelligence and machine learning have the potential to dramatically change the way we do business, the economy, and society in the coming decades. The authors argue that these advances will have a profound impact on the future of work, education, health care, transportation, energy, finance, government, law, media, entertainment, politics, science, technology and the environment. They also predict that the impact of these changes will be felt most acutely by those who are least able to adapt to them, such as the elderly, people with disabilities, low-skilled workers, women, racial and ethnic min

### Temperature


In [ ]:

args = {"do_sample":True, "max_length":  300, "top_k":0}

temperatures = [0.5, 0.75, 1, 1.25, 1.5]

results = []

for temperature in temperatures:
    args["temperature"] = temperature

    result = generate_text(tokenizer,model, device,input_text, **args)

    results.append(result)


result_df = pd.DataFrame(results)
display (result_df)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generati

,prompt,generated_text,do_sample,max_length,top_k,temperature
0,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence,"" in Artificial Intelligence: A Modern Approach, S. M. Stern, ed. Springer-Verlag, pp. 471-482.\n\n[6] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[7] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[8] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[9] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[10] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[11] See, for example, the recent paper by the authors of this paper on th...",True,300,0,0.50
1,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence for the Future of Human Health and Well-Being,"" whose results were posted at the journal Health Affairs on Wednesday.\n\nThe researchers examined the effect of self-driving cars in the United States over the next five years. The paper says that self-driving cars are ""likely to play an even more important role in health and well-being over the next several decades"" than they do today.\n\nIndeed, the military, police and emergency crews are already using self-driving cars to ensure the safety of their operations. And Google is building a fleet of cars that can drive themselves.\n\nNow, it appears that the federal government is about to take a significant step in this direction.\n\nThe National Highway Traffic Safety Administration (NHTSA) has approved a request to begin testing self-driving cars on public roads throughout the country. The car will be able to operate at up to 60 mph and will be able to detect pedestria...",True,300,0,0.75
2,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence for the National Security of the United States"" read Jim Holleran, two-star Army General and head of the Defense Advanced Research Projects Agency. Holleran was giving the title presentation at the RAND conference in California this week.\n\nNo major investor seemed to notice, but none, perhaps, better than Brigadier General Michael Dunlavey, former Deputy Assistant Secretary of Defense for Science, Mathematics, and Intelligence. In September 2003, as head of the Defense Warrior Development Program, an elite program that in the former Soviet Union provided limited military training to young men and women, Dunlavey oversaw the creation of the Advanced Research Projects Agency complex within Kirtland Air Force Base in New Mexico.\n\nAlong with funding his much-hyped Defense Enhancement Program, like that 175,000 women serve in the army he spent much of the first 10 years of the 21st century making the mind-numbing des...",True,300,0,1.00
3,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence."") An NPR entry notes that previous passages had detailed how macroeconomic outcomes could change as a consequence of smarter robots. Lt. Cmdr. Eleftheria Olgheard uses robots in white paper onboard commercial 16-story super cruise liner Tribal If top AI chips systems make the system less vulnerable financially and more hard to hack, why even bother the Indian Armor? But there is an army XII658, which NurGam considers, a steampunkgish Jan Mary recommended Marvel Cthulhu Royal Guard personality portfolio, through 1 füte

### Top k

In [ ]:
args = {"do_sample":True, "max_length":  300,"top_k":50}

# max_new_tokens=n_steps,
result = generate_text(tokenizer,model, device,input_text, **args)

result_df = pd.DataFrame([result])
display (result_df)

generate_text(tokenizer,model, device,input_text,**args )

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,top_k
0,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence,"" by Peter Norvig of Stanford University, is another recent book that focuses on the impact of artificial intelligence. It provides the usual suspects, but with an analysis which points out that the issues and impact of AI are rapidly evolving and with increasingly clear negative implications for the future of humanity.\n\nAll these books should be taken as useful indicators of the general shape of the new era, which was forecast by the founders of the space race who were well informed about what was coming, and whom we might have expected to be the most optimistic of optimists about human destiny. The new millennium will be one of massive human empowerment and transformation, with immense benefits to all.\n\nHowever, when this moment of profound possibility was finally reached, it was quickly replaced by a feeling of dread that seems to have no cause. The fear that, without any change, we would have ended up in a s...",True,300,50


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'prompt': 'The Implications of Recent Developments in Artificial Intelligence',
 'generated_text': "The Implications of Recent Developments in Artificial Intelligence and Learning Theory for Education and Teaching\n\nRochester Institute of Technology\n\nMay 1 – 5, 2017\n\nDownload lecture slides\n\nPre-registering for the conference is strongly encouraged. To register, please visit the event page.\n\nJoin our conversation on Facebook!\n\nAbout Rochester Tech\n\nRochester Institute of Technology (RIT) is one of the world's largest and most comprehensive research universities. A comprehensive mix of technology, engineering, and liberal arts disciplines allow RIT to excel in education, research, and public service. We are a community of over 42,000 students and alumni, and more than 300,000 in its research, teaching, and community settings each year. For more information about RIT and to join our community, please visit<|endoftext|>",
 'do_sample': True,
 'max_length': 300,
 'top_k': 50}

Top p

In [ ]:
args = {"do_sample":True, "max_length":  300,"top_p":0.9}

# max_new_tokens=n_steps,
result = generate_text(tokenizer,model, device,input_text, **args)

result_df = pd.DataFrame([result])
display (result_df)

generate_text(tokenizer,model, device,input_text,**args )

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,top_p
0,The Implications of Recent Developments in Artificial Intelligence,The Implications of Recent Developments in Artificial Intelligence\n\n(R&R)\n\nMachine Learning: What's New and What Is Good?\n\nby\n\nAndrew Ng\n\n(R&R)\n\nThe Future of Artificial Intelligence\n\nby\n\nSteven Johnson\n\n(R&R)\n\nThe New Machine Learning (R&R)\n\nby\n\nSteven Johnson\n\n(R&R)\n\nMachine Learning and Data Mining\n\nby\n\nStephen Wolfram\n\n(R&R)\n\nMachine Learning and Decision Making\n\nby\n\nRishabh Khurana\n\n(R&R)\n\nThe Art of Data Mining and Natural Language Processing\n\nby\n\nJeff Dean\n\n(R&R)\n\nMachine Learning for Visualization and Computer Vision\n\nby\n\nKai-Fu Lee\n\n(R&R)\n\nDeep Learning\n\nby\n\nAlex Graves\n\n(R&R)\n\nMachine Learning for Financial Data and Risk Analysis\n\nby\n\nBen Goertzel\n\n(R&R)\n\nApplied Machine Learning\n\nby\n\nAndrew Ng\n\n(R&R)\n\nData Mining and Natural Language Processing: The Definitive Guide\n\nby\n\nJohn Bickers\n\n(R&R)\n\nMachine Learning for Image Understanding and Classification\n\nby\n\nAkihiko N,True,300,0.9


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'prompt': 'The Implications of Recent Developments in Artificial Intelligence',
 'generated_text': 'The Implications of Recent Developments in Artificial Intelligence for Business\n\nA recent survey by the Pew Research Center found that nearly half of the companies surveyed said that AI is one of the top three biggest threats to business over the next five years, more than terrorism or foreign exchange rate fluctuations. This is particularly concerning because AI, which is defined as a system that can operate with little or no human intervention, represents an unprecedented combination of power, precision, and efficiency. Although its potential for impact on our businesses is uncertain, AI could be a game-changing innovation that will have a profound impact on our world.\n\nWhat Is AI and What Is the Potential Impact?\n\nA major goal of AI is to do what it was created to do. One way to understand this is that AI creates what is commonly referred to as "intelligence," or the ability to

## Tools for the task
* Some great working examples can be found in the file **05_text_generation.ipynb** from the [Natural Language Processing with Transformers GitHub Repository](https://github.com/nlp-with-transformers/notebooks).
* There is a variety of models on Hugging Face that allow you to try out different models for your task. It is strongly recommended that you review these and look at the model cards in each case. Also, you can test your text in the Hosted inference API on the right hand side of each model.
* If you want to consider execution time of text generation with different decoding configurations or prompts, you can consider using [Python timing functions](https://realpython.com/python-timer/)

# Submission of your Assignment
Submit your main Jupyter notebook file and your articles for your assignment through Brightspace in the same manner as you have before. Near the top of the file, provide a link to your GitHub repository for the task which contains your input and output files.